# Deposit Attrition EDA — v10 · What the work is worth

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

v9 established three things and exposed one problem.

**Established.** At-risk dollars are extremely concentrated (Gini 0.928; the top 1% of departing
clients hold 64.5%). The model is *better* where the money is (AUC 0.873 in balance deciles 9–10
against 0.734 in 1–2). And the break-even RM save rate is **0.10%** — one save per thousand
conversations — so the business case does not rest on any assumption about how persuasive an RM is.

**The problem.** The money leaves at `rel_m −8 … −6`; the queue arrives at −1 to −2.

| rel_m | share of relationship still on book | defendable |
|---|---|---|
| −12 | 0.979 | $10.01bn |
| −9 | 0.901 | $9.21bn |
| **−7** | **0.608** | — |
| **−6** | **0.346** | **$3.54bn** |
| −2 | 0.183 | $1.87bn |
| −1 | 0.088 | $897m |

Two months carry 51 points of the decline, against a stayer line flat at 1.02–1.06. **Lead time,
not precision, is the binding constraint**, and every previous run optimised precision because
that is what a client-count queue rewards.

## What v10 does

| § | |
|---|---|
| 2 | A third label — **`D_money_move`**, the month the balance actually moves rather than the month the account closes. Built with an explicit guard against the circularity that voided v4 |
| 3 | **Leakage audit.** Rolling-origin window table, and a label-permutation control that must collapse to AUC 0.5. Nothing downstream is defendable without this |
| 4 | **Four feature sets** — deposit only, payments only, both, all — defined by construction and held identical across every comparison |
| 5 | Train and score every feature set at **five horizons**, rolling origin |
| 6 | **Isotonic recalibration.** v9's top decile was 41% under-predicted and that is exactly where the queue operates |
| 7 | **The savings engine** — feature set × alert volume × save rate. Success applies only to true attriters; the balance freezes at the moment of a successful call |
| 8 | **The lead-time ceiling** — what the same clients would have been worth caught L months earlier. Separates *timing* from *detection* |
| 9 | Robustness — jackknife the top 10 clients, the B pool, and the destination over-index confound |

## The claim this notebook is built to support

> Even at a **1% save rate**, a payment-informed queue retains materially more than a deposit-only
> queue at the same alert volume — and the difference is the value of this work.

## 0 · Configuration

In [ ]:
# =====================================================================
# 0 · CONFIGURATION — v10
# =====================================================================
from pathlib import Path

HDFS_V2  = "hdfs://nameservice1/user/pk36814/attrition_v2"
HDFS_V6  = "hdfs://nameservice1/user/pk36814/attrition_v6"
HDFS_V7  = "hdfs://nameservice1/user/pk36814/attrition_v7"
HDFS_V8  = "hdfs://nameservice1/user/pk36814/attrition_v8"
HDFS_V9  = "hdfs://nameservice1/user/pk36814/attrition_v9"
HDFS_DIR = "hdfs://nameservice1/user/pk36814/attrition_v10"
OUT_DIR  = Path("/projects/DSI/sa15474/repos/pkg/eda/attrition_v10")
# TRAP: pathlib collapses hdfs://host/p -> hdfs:/host/p. Local Path and HDFS
# string stay separate variables and are never mixed.

DATE_START, DATE_END = "2024-01-01", "2026-07-31"
MAX_ROWS, SEED = 60, 20260909

# ── carried unchanged from v7-v9 ──────────────────────────────────────
CHG_LAG_FAR, CHG_LAG_NEAR, CHG_MIN_OBS, MIN_REF = -6, -4, 2, 1.0
PEER_MIN_N, DD_CLIP, BAL_FLOOR = 50, (0.01, 100.0), 1_000.0
ORIGIN_START_OFF, MAX_ORIGINS = 18, 24
NEG_SAMPLE, MIN_TRAIN_POS = 0.15, 200
L2, IRLS_MAX_IT, IRLS_TOL = 2.0, 60, 1e-9
MAX_COLLECT_ROWS, COOLDOWN_M = 3_500_000, 3

# ── §2 the new label ──────────────────────────────────────────────────
MOVE_FRAC   = 0.50     # "the money moved" = balance below half its own
                       # trailing-12 median...
MOVE_HOLD   = 2        # ...and it stays there this many months
MOVE_MIN_HIST = 9      # months of balance history required before a move
                       # can be declared
# GUARD against the circularity that voided v4: a client is only AT RISK
# for D at month t if it has NOT already moved. Without this, bal_live's
# own dd predicts a label defined on bal_live and the deposit-only
# feature set wins by construction.
MOVE_ATRISK_FRAC = 0.50

# ── §5 horizons ───────────────────────────────────────────────────────
HORIZON_GRID = [1, 3, 6, 9, 12]
PRIMARY_H    = 6
PRIMARY_DEF  = "A_full_exit"
LABELS_RUN   = ["A_full_exit", "D_money_move"]

# ── §7 the savings engine ─────────────────────────────────────────────
CAPACITY    = [50, 100, 250, 500, 1000, 2500, 5000]
QUEUE_K     = 1000
P_SAVE_GRID = [0.01, 0.05, 0.10, 0.20, 0.50]
P_SAVE_BASE = 0.10
RM_COST_PER_CALL = 250.0
ANNUALISE   = 12.0/7.0
VALUE_ALPHAS = [0.0, 0.5, 0.75, 1.0]
ALPHA_BASE   = 0.75

# ── §8 the lead-time ceiling ──────────────────────────────────────────
LEAD_GRID = [1, 2, 3, 4, 6, 9, 12]

# ── §9 robustness ─────────────────────────────────────────────────────
JACKKNIFE_TOP = [0, 10, 50, 100]   # drop the N largest at-risk clients

RUN_LABELS, RUN_AUDIT, RUN_FIT = True, True, True
RUN_SAVINGS, RUN_LEAD, RUN_ROBUST, RUN_REPORT = True, True, True, True
HTML_NAME = "PKG_Attrition_v10_Value.html"


In [ ]:
# =====================================================================
# 1 · IMPORTS, SESSION, HELPERS, CHARTS
# =====================================================================
import warnings, time, math, json, datetime as dt
import numpy as np, pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)
spark = (SparkSession.builder.appName("pkg_attrition_eda_v10")
         .config("spark.sql.shuffle.partitions", "800")
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         # KEEP OFF. PySpark 3.3.2's arrow path references np.object0/np.bool8,
         # both removed in numpy 2.0.
         .config("spark.sql.autoBroadcastJoinThreshold", str(64*1024*1024))
         .enableHiveSupport().getOrCreate())
pd.set_option("display.max_columns", 400); pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

HAVE_MPL = True
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter
    plt.rcParams.update({
        "figure.dpi": 120, "savefig.dpi": 120, "font.size": 9,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": "#E5E7EB", "grid.linewidth": .7,
        "axes.edgecolor": "#9AA1AC", "axes.labelcolor": "#16181D",
        "text.color": "#16181D", "xtick.color": "#6B7280", "ytick.color": "#6B7280",
        "figure.facecolor": "white", "axes.facecolor": "white"})
except Exception as e:
    HAVE_MPL = False
    print(f"  matplotlib unavailable ({e}) — charts degrade to tables")
print(f"numpy {np.__version__} · pandas {pd.__version__} · spark {spark.version} · "
      f"charts {'on' if HAVE_MPL else 'OFF'}")

ACC, ACC2, GOOD, WARN, GREY, INK = "#C1440E", "#4A6FA5", "#2F6F4E", "#B8860B", "#9AA1AC", "#16181D"
FSCOL = {"deposit_only": GREY, "payment_only": ACC2, "both": GOOD, "all_features": ACC,
         "incumbent": "#7B5EA7"}

def hp(n): return f"{HDFS_DIR.rstrip('/')}/{n}"
def v2(n): return f"{HDFS_V2.rstrip('/')}/{n}"
def v6(n): return f"{HDFS_V6.rstrip('/')}/{n}"
def v7(n): return f"{HDFS_V7.rstrip('/')}/{n}"
def v8(n): return f"{HDFS_V8.rstrip('/')}/{n}"
def v9(n): return f"{HDFS_V9.rstrip('/')}/{n}"
def exists(p):
    try: spark.read.parquet(p).limit(1).count(); return True
    except Exception: return False
def pct(a, b): return float(a)/float(b) if b else float("nan")
def _dec(s):
    o = s
    for c, t in s.dtypes:
        if t.startswith("decimal"): o = o.withColumn(c, F.col(c).cast("double"))
    return o
def usd(v):
    if v is None or (isinstance(v, float) and not np.isfinite(v)): return "—"
    a = abs(v)
    if a >= 1e9: return f"${v/1e9:,.2f}bn"
    if a >= 1e6: return f"${v/1e6:,.1f}m"
    if a >= 1e3: return f"${v/1e3:,.0f}k"
    return f"${v:,.0f}"
def usd_col(df, cols):
    d = df.copy()
    for c in ([cols] if isinstance(cols, str) else cols):
        if c in d.columns: d[c] = d[c].map(usd)
    return d
def disp(o, title=None, n=None, save=None):
    n = MAX_ROWS if n is None else n
    out = _dec(o).limit(n).toPandas() if hasattr(o, "toPandas") else (
        o.copy() if isinstance(o, pd.DataFrame) else pd.DataFrame(o))
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                     f"margin:10px 0 2px;color:#111'>{title}"
                     f"<span style='font-weight:400;color:#888'> &middot; {len(out)} rows"
                     f"</span></div>"))
    display(out); return out
def kv(pairs, title=None, save=None):
    items = list(pairs.items()) if isinstance(pairs, dict) else list(pairs)
    labs = [k for k, _ in items]
    d = sorted({k for k in labs if labs.count(k) > 1})
    if d: raise ValueError(f"kv(): duplicate labels {d}")
    return disp(pd.DataFrame({"metric": labs, "value": [v for _, v in items]}),
                title=title, n=len(items), save=save)
def collect_pd(sdf, label="", max_rows=None):
    max_rows = MAX_COLLECT_ROWS if max_rows is None else max_rows
    n = sdf.count()
    if n > max_rows: raise RuntimeError(f"{label}: {n:,} > {max_rows:,}")
    t0 = time.time(); out = _dec(sdf).toPandas()
    print(f"  collected {label}: {n:,} x {out.shape[1]} in {time.time()-t0:,.0f}s")
    return out

# ── charts ────────────────────────────────────────────────────────────
def _fin(fig, title, sub=None, save=None):
    if sub: fig.text(0.005, 0.965, sub, fontsize=8, color="#6B7280", va="top")
    fig.suptitle(title, fontsize=11, fontweight="bold", x=0.005, ha="left", y=1.0)
    fig.tight_layout(rect=[0, 0, 1, 0.94 if sub else 0.96])
    if save: fig.savefig(OUT_DIR / f"{save}.png", bbox_inches="tight")
    display(fig); plt.close(fig)

def bar_grouped(df, x, series, title, sub=None, ylab="", fmt=usd, save=None,
                colors=None, figsize=(9.5, 4.2), log=False):
    """df indexed by x, one column per series."""
    if not HAVE_MPL: return disp(df.reset_index(), title=title)
    fig, ax = plt.subplots(figsize=figsize)
    n = len(series); w = 0.8/n
    idx = np.arange(len(df))
    for i, s in enumerate(series):
        c = (colors or {}).get(s, None)
        b = ax.bar(idx + (i-(n-1)/2)*w, df[s].values, w, label=s, color=c)
        for r, v in zip(b, df[s].values):
            if np.isfinite(v):
                ax.text(r.get_x()+r.get_width()/2, v, fmt(v), ha="center", va="bottom",
                        fontsize=7, rotation=0)
    ax.set_xticks(idx); ax.set_xticklabels([str(i) for i in df.index])
    ax.set_xlabel(x); ax.set_ylabel(ylab)
    if log: ax.set_yscale("log")
    ax.legend(frameon=False, fontsize=8, ncol=min(n, 4))
    ax.margins(y=0.18)
    _fin(fig, title, sub, save)

def heat(df, title, sub=None, fmt=usd, save=None, figsize=(9.5, 4.4),
         xlab="", ylab="", cmap="OrRd"):
    if not HAVE_MPL: return disp(df.reset_index(), title=title)
    fig, ax = plt.subplots(figsize=figsize)
    V = df.values.astype(float)
    im = ax.imshow(V, cmap=cmap, aspect="auto")
    ax.set_xticks(range(df.shape[1])); ax.set_xticklabels(df.columns, fontsize=8)
    ax.set_yticks(range(df.shape[0])); ax.set_yticklabels(df.index, fontsize=8)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.grid(False)
    mx = np.nanmax(V)
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            v = V[i, j]
            if np.isfinite(v):
                ax.text(j, i, fmt(v), ha="center", va="center", fontsize=7.5,
                        color=("white" if v > 0.62*mx else "#16181D"))
    fig.colorbar(im, ax=ax, shrink=.85, pad=.015)
    _fin(fig, title, sub, save)

def lines(df, title, sub=None, ylab="", xlab="", fmt=None, save=None,
          colors=None, figsize=(9.5, 4.2), logx=False, annotate_last=True):
    if not HAVE_MPL: return disp(df.reset_index(), title=title)
    fig, ax = plt.subplots(figsize=figsize)
    for c in df.columns:
        ax.plot(df.index, df[c].values, marker="o", ms=4.5, lw=2,
                color=(colors or {}).get(c, None), label=str(c))
        if annotate_last and np.isfinite(df[c].values[-1]):
            ax.annotate(str(c), (df.index[-1], df[c].values[-1]), fontsize=7.5,
                        xytext=(5, 0), textcoords="offset points", va="center")
    if logx: ax.set_xscale("log"); ax.set_xticks(list(df.index)); \
             ax.set_xticklabels([f"{i:,}" for i in df.index], fontsize=8)
    if fmt: ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: fmt(v)))
    ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    ax.legend(frameon=False, fontsize=8, ncol=min(len(df.columns), 4))
    _fin(fig, title, sub, save)

def scatter(pts, title, sub=None, xlab="", ylab="", save=None, figsize=(9, 5),
            fmtx=None, fmty=None):
    """pts: list of dicts(x, y, s, label, color, tag)"""
    if not HAVE_MPL: return disp(pd.DataFrame(pts), title=title)
    fig, ax = plt.subplots(figsize=figsize)
    seen = set()
    for p in pts:
        lab = p.get("label")
        ax.scatter(p["x"], p["y"], s=p.get("s", 60), color=p.get("color"),
                   alpha=.82, edgecolor="white", linewidth=.8,
                   label=(lab if lab not in seen else None))
        seen.add(lab)
        if p.get("tag"):
            ax.annotate(p["tag"], (p["x"], p["y"]), fontsize=7,
                        xytext=(6, 4), textcoords="offset points")
    if fmtx: ax.xaxis.set_major_formatter(FuncFormatter(lambda v, q: fmtx(v)))
    if fmty: ax.yaxis.set_major_formatter(FuncFormatter(lambda v, q: fmty(v)))
    ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    ax.legend(frameon=False, fontsize=8)
    _fin(fig, title, sub, save)

# ── model ─────────────────────────────────────────────────────────────
def auc(y, s):
    y = np.asarray(y, float); s = np.asarray(s, float)
    ok = np.isfinite(s) & np.isfinite(y); y, s = y[ok], s[ok]
    n1 = float(y.sum()); n0 = float(len(y)-n1)
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum() - n1*(n1+1)/2.0)/(n1*n0))

def logit_irls(X, y, l2=L2, mi=IRLS_MAX_IT, tol=IRLS_TOL):
    X = np.asarray(X, np.float64); y = np.asarray(y, np.float64)
    b = np.zeros(X.shape[1]); R = l2*np.eye(X.shape[1]); R[0, 0] = 0.0
    for _ in range(mi):
        eta = np.clip(X @ b, -30, 30); mu = 1/(1+np.exp(-eta))
        w = np.maximum(mu*(1-mu), 1e-6); z = eta + (y-mu)/w; XtW = X.T*w
        try: bn = np.linalg.solve(XtW @ X + R, XtW @ z)
        except np.linalg.LinAlgError:
            bn = np.linalg.lstsq(XtW @ X + R, XtW @ z, rcond=None)[0]
        if np.max(np.abs(bn-b)) < tol: b = bn; break
        b = bn
    return b

def fit_spec(tr, cols, l2=L2, s=NEG_SAMPLE):
    X = tr[cols].to_numpy(np.float64); y = tr["y"].to_numpy(np.float64)
    keep = X.std(axis=0) > 1e-9
    ck = [c for c, k in zip(cols, keep) if k]
    if not ck or y.sum() < 2: return None
    Xk = X[:, keep]; mu = Xk.mean(0); sd = Xk.std(0)
    b = logit_irls(np.column_stack([np.ones(len(Xk)), (Xk-mu)/sd]), y, l2)
    return dict(cols=ck, beta=b[1:]/sd,
                b0=float(b[0]-float(np.sum(b[1:]*mu/sd))) + math.log(s))

def predict_p(sp, df):
    if sp is None: return np.full(len(df), np.nan)
    eta = sp["b0"] + df[sp["cols"]].to_numpy(np.float64) @ sp["beta"]
    return 1.0/(1.0 + np.exp(-np.clip(eta, -30, 30)))

def pav(x, y, w=None, nbins=200):
    """Isotonic fit by pool-adjacent-violators, on QUANTILE BINS.

    Binning first is not an approximation shortcut, it is what makes this
    usable: a raw PAV that deletes from a python list is O(n^2), and the
    calibration slice here is ~150k rows — it does not return. Binned to a
    few hundred points and merged with a stack, it is O(nbins)."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) == 0: return np.array([0.0]), np.array([0.0])
    nb = int(min(nbins, max(2, len(x)//50)))
    edges = np.unique(np.quantile(x, np.linspace(0, 1, nb+1)))
    if len(edges) < 3:
        return (np.array([float(x.min()), float(x.max())]),
                np.array([float(y.mean())]*2))
    idx = np.clip(np.searchsorted(edges, x, side="right")-1, 0, len(edges)-2)
    g = (pd.DataFrame({"b": idx, "x": x, "y": y}).groupby("b")
         .agg(x=("x", "mean"), y=("y", "mean"), w=("y", "size")).sort_values("x"))
    sx, sy, sw = [], [], []
    for xi, yi, wi in zip(g.x.values, g.y.values, g.w.values.astype(float)):
        sx.append(float(xi)); sy.append(float(yi)); sw.append(float(wi))
        while len(sy) > 1 and sy[-2] > sy[-1]:
            nw = sw[-2] + sw[-1]
            sy[-2] = (sy[-2]*sw[-2] + sy[-1]*sw[-1])/nw
            sw[-2] = nw; sx[-2] = sx[-1]
            sy.pop(); sw.pop(); sx.pop()
    return np.asarray(sx), np.asarray(sy)

def apply_iso(knots, p):
    kx, ky = knots
    p = np.asarray(p, float)
    # a calibration slice with no positives collapses the map to zero, which
    # would silently zero every expected-value figure. Fall back to raw.
    if len(ky) == 0 or not np.isfinite(ky).any() or np.nanmax(ky) <= 0:
        return p
    return np.interp(p, kx, ky, left=ky[0], right=ky[-1])

_F = OUT_DIR / "FINDINGS_v10.csv"
FINDINGS = pd.read_csv(_F).to_dict("records") if _F.exists() else []
def note(qid, q, a, d=""):
    global FINDINGS
    FINDINGS = [f for f in FINDINGS if f["id"] != qid]
    FINDINGS.append(dict(id=qid, question=q, answer=str(a), detail=str(d)))
    pd.DataFrame(FINDINGS).to_csv(_F, index=False)
print("helpers ready")


## 2 · A third label — the month the money moves

`A_full_exit` is the month the last account goes non-live. The decay curve says the money left
**six to eight months earlier**. Predicting closure is predicting the aftermath.

```
D_money_move(c) = first month t where
      bal_live(c, t) < 0.50 × median( bal_live over t-11 … t )
   and that holds for 2 consecutive months
   and the client has ≥ 9 months of balance history
```

### The circularity guard
v4 was voided because peer deciles built on `bal_live` made `bal_live` uninformative by
construction. The mirror risk here is the opposite and worse: a label defined **on** `bal_live`
would be trivially predicted **by** `bal_live`, and the deposit-only feature set would win by
construction rather than by merit.

**The guard:** a client is only *at risk* for `D` at month `t` if it has **not already moved** —
`bal_now(t) ≥ 0.50 × bal_med12(t)`. Combined with a forward-only label window, the model is asked
to predict a move that has not begun, from features measured before it began. That is a real
prediction problem.

It is still not perfectly clean — a gradual decline is partly visible at `t` — so §5 reports
`A_full_exit` and `D_money_move` side by side and the deposit-only advantage on `D` is read with
that in mind.

In [ ]:
# =====================================================================
# 2 · LABELS — A, B, and the new D                       [OUTPUT BLOCK 1]
# =====================================================================
t0 = time.time()
cust_month = spark.read.parquet(v2("panel_customer_month")).filter(F.col("ym") >= DATE_START[:7])
YMMAP = cust_month.select("ym", "m_idx").distinct().persist(StorageLevel.DISK_ONLY)
M_MIN, M_MAX = [int(x) for x in cust_month.agg(F.min("m_idx"), F.max("m_idx")).collect()[0]]
print(f"  m_idx {M_MIN}-{M_MAX} (ABSOLUTE — every month constant is an offset)")
lab = spark.read.parquet(v6("labels_customer")).persist(StorageLevel.DISK_ONLY)
BAL = spark.read.parquet(v9("balance")).persist(StorageLevel.DISK_ONLY)  # built in v9 §2

if RUN_LABELS:
    wf = Window.partitionBy("cust_pwr_id").orderBy("m_idx").rangeBetween(0, MOVE_HOLD-1)
    MV = (BAL.select("cust_pwr_id", "m_idx", "bal_now", "bal_med12", "bal_n12")
          .withColumn("low", ((F.col("bal_n12") >= MOVE_MIN_HIST) &
                              (F.col("bal_med12") > BAL_FLOOR) &
                              (F.col("bal_now") < MOVE_FRAC*F.col("bal_med12"))).cast("int"))
          .withColumn("run", F.sum("low").over(wf))
          .withColumn("obs_fwd", F.count("*").over(wf))
          .withColumn("is_move", ((F.col("run") == MOVE_HOLD) &
                                  (F.col("obs_fwd") == MOVE_HOLD)).cast("int")))
    DLAB = (MV.filter(F.col("is_move") == 1).groupBy("cust_pwr_id")
            .agg(F.min("m_idx").alias("q_D_money_move")))
    DLAB.write.mode("overwrite").parquet(hp("labels_D"))
    DLAB = spark.read.parquet(hp("labels_D")).persist(StorageLevel.DISK_ONLY)

    LAB = lab.join(DLAB, "cust_pwr_id", "left").persist(StorageLevel.DISK_ONLY)
    cmp_ = (LAB.filter(F.col("q_A_full_exit").isNotNull())
            .select("cust_pwr_id", "q_A_full_exit", "q_D_money_move",
                    (F.col("q_A_full_exit")-F.col("q_D_money_move")).alias("gap")))
    n_a = LAB.filter(F.col("q_A_full_exit").isNotNull()).count()
    n_d = LAB.filter(F.col("q_D_money_move").isNotNull()).count()
    n_both = cmp_.filter(F.col("q_D_money_move").isNotNull()).count()
    g = cmp_.filter(F.col("gap").isNotNull()).agg(
            F.expr("percentile_approx(gap, 0.25)").alias("p25"),
            F.expr("percentile_approx(gap, 0.5)").alias("p50"),
            F.expr("percentile_approx(gap, 0.75)").alias("p75"),
            F.avg((F.col("gap") > 0).cast("double")).alias("share_before")).collect()[0]
    kv([("A_full_exit, qualified attriters", f"{n_a:,}"),
        ("D_money_move, clients with a move", f"{n_d:,}"),
        ("A attriters that also have a D", f"{n_both:,} ({pct(n_both, n_a):.1%})"),
        ("months D precedes A — p25 / median / p75",
         f"{g['p25']:.0f} / {g['p50']:.0f} / {g['p75']:.0f}"),
        ("share where D strictly precedes A", f"{g['share_before']:.1%}"),
        ("block 2 wall (s)", round(time.time()-t0))],
       title="2a &middot; <b>The money moves before the account closes.</b> If the median gap is "
             "6&ndash;8 months, D is the event worth predicting and A is its aftermath",
       save="v10_label_D")

    GAP = collect_pd(cmp_.filter(F.col("gap").isNotNull()).select("gap"), "A-D gaps")
    if HAVE_MPL and len(GAP):
        gg = GAP[GAP.gap.between(-6, 24)].gap.value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(9.5, 3.6))
        ax.bar(gg.index, gg.values, color=[ACC if i > 0 else GREY for i in gg.index])
        ax.axvline(0, color=INK, lw=1, ls=":")
        ax.set_xlabel("months the money move precedes the account closure")
        ax.set_ylabel("clients")
        _fin(fig, "2b · How far ahead of the closure does the money actually move?",
             f"median {g['p50']:.0f} months · {g['share_before']:.1%} strictly before",
             save="v10_gap_hist")
    note("LABELD", "Does the money move before the account closes?",
         f"median {g['p50']:.0f} months earlier, {g['share_before']:.1%} strictly before",
         "If so, A_full_exit is the aftermath and D_money_move is the event a retention queue "
         "should be aimed at.")


## 3 · Leakage audit

Everything downstream is a dollar claim, so the separation between training and test has to be
demonstrated rather than asserted. Three checks:

1. **Window table.** For every fold, the maximum training month, the test month, and the month at
   which the training labels resolve. The rule is `train t ≤ T − H`, so **every training label has
   already resolved before the test month begins**.
2. **Feature-window audit.** Every feature is built from `t` and the window `t−6 … t−4`; the peer
   median is cross-sectional within the same calendar month, which exists at scoring time; the
   frozen decile comes from the client's first three months. No column reads forward.
3. **Label permutation control.** Refit on shuffled training labels and score the untouched test
   month. **AUC must collapse to ≈ 0.50.** If it does not, something in the pipeline is carrying
   the answer across.

In [ ]:
# =====================================================================
# 3 · LEAKAGE AUDIT                                      [OUTPUT BLOCK 2]
# =====================================================================
ORIGINS = list(range(M_MIN + ORIGIN_START_OFF, M_MAX - min(HORIZON_GRID) + 1))
ORIGINS = [t for t in ORIGINS if t <= M_MAX - PRIMARY_H]
assert 1 <= len(ORIGINS) <= MAX_ORIGINS, (
    f"{len(ORIGINS)} origins — m_idx is ABSOLUTE ({M_MIN}-{M_MAX}); "
    f"ORIGIN_START_OFF is an OFFSET")
print(f"  origins m_idx {ORIGINS[0]}-{ORIGINS[-1]} ({len(ORIGINS)} folds)")

if RUN_AUDIT:
    rows = []
    for H in HORIZON_GRID:
        for T in ORIGINS:
            if T + H > M_MAX: continue
            rows.append(dict(horizon=H, test_month=T, train_max_month=T-H,
                             train_labels_resolve_by=T,
                             test_labels_resolve_by=T+H,
                             gap_months=H,
                             ok=(T-H) + H <= T))
    W = pd.DataFrame(rows)
    disp(W[W.horizon == PRIMARY_H],
         title=f"3a &middot; <b>Rolling-origin windows at H={PRIMARY_H}.</b> Training stops at "
               f"<code>T&minus;H</code> so every training label resolves by the test month; the "
               f"test labels resolve afterwards and are never seen in fitting",
         n=30, save="v10_windows")
    assert W.ok.all(), "a fold would train on labels that had not resolved"
    kv([("folds per horizon", len(ORIGINS)),
        ("horizons", str(HORIZON_GRID)),
        ("total model fits (4 feature sets x horizons x folds)",
         4*sum(1 for H in HORIZON_GRID for T in ORIGINS if T+H <= M_MAX)),
        ("training rows sampled at", NEG_SAMPLE),
        ("test rows", "FULL book — no sampling, so precision is exact"),
        ("window assertion", "PASSED — no fold trains on an unresolved label")],
       title="3b &middot; Design", save="v10_audit_design")


## 4 · Four feature sets

Defined by construction, held identical across every comparison in this notebook. Each is a strict
superset of the one before except that `deposit_only` and `payment_only` are disjoint.

| Set | Contents | The question it answers |
|---|---|---|
| **`deposit_only`** | `bal_live` dd, balance level / volatility / drawdown / top-account share, live-account trajectory, closed share | What the bank can do with the deposit system alone — the honest counterfactual to this whole programme |
| **`payment_only`** | The 20 payment dd features. **No balance at all** | Can payments do it without ever looking at the ledger? |
| **`both`** | `deposit_only` + `payment_only` | The v7 model, roughly |
| **`all_features`** | `both` + inbound institutions, outbound counterparty churn, recurring-series breakage | Everything v8b shipped |

`incumbent` — the 30% balance rule as a monthly flag — is carried as the do-nothing reference.

In [ ]:
# =====================================================================
# 4 · FEATURE SETS                                       [OUTPUT BLOCK 3]
# =====================================================================
RISK = spark.read.parquet(v9("risk_set_v9")).persist(StorageLevel.DISK_ONLY)
ALLC = RISK.columns
NEWP = ("cptyn_", "cptya_", "fin_out2", "fin_in", "tim_", "railmix", "conc_",
        "selfpay", "acc_", "bl_")
def _pair(ld):
    return sorted(set(ld + [f"md_{c[3:]}" for c in ld if f"md_{c[3:]}" in ALLC]))
V7_LD = sorted([c for c in ALLC if c.startswith("ld_")
                and not any(c.startswith("ld_"+p) for p in NEWP)])
DEP_CORE = [c for c in V7_LD if c in ("ld_bal_live",)]
PAY_CORE = [c for c in V7_LD if c not in DEP_CORE]
def blk(*pfx):
    return _pair(sorted([c for c in ALLC if any(c.startswith("ld_"+p) for p in pfx)]))

DEPOSIT = _pair(DEP_CORE) + blk("bl_") + blk("acc_")
PAYMENT = _pair(PAY_CORE)
BOTH    = sorted(set(DEPOSIT + PAYMENT))
EXTRA   = blk("fin_in") + blk("cptya_out") + blk("fin_out2") + \
          sorted([c for c in ALLC if "rec_" in c and c.startswith(("ld_", "md_"))])
ALLF    = sorted(set(BOTH + EXTRA))
FSETS = {"deposit_only": DEPOSIT, "payment_only": PAYMENT,
         "both": BOTH, "all_features": ALLF}
FS_ORDER = ["deposit_only", "payment_only", "both", "all_features"]
disp(pd.DataFrame([dict(feature_set=k, n_features=len(v),
                        example=", ".join([c[3:] for c in v if c.startswith("ld_")][:4]) + " …")
                   for k, v in FSETS.items()]),
     title="4a &middot; The four sets, by construction", save="v10_feature_sets")
assert not (set(DEPOSIT) & set(PAYMENT)), "deposit and payment sets must be disjoint"

NEED = sorted(set(["cust_pwr_id", "m_idx", "event_A", "event_B",
                   "bar_now", "bar_med12", "bar_peak12"] + ALLF) & set(ALLC))
DLAB2 = spark.read.parquet(hp("labels_D")) if exists(hp("labels_D")) else None
RS = RISK.select(*NEED)
if DLAB2 is not None:
    RS = RS.join(DLAB2.withColumnRenamed("q_D_money_move", "event_D"), "cust_pwr_id", "left")
else:
    RS = RS.withColumn("event_D", F.lit(None).cast("int"))
# at-risk guard for D: the client must not have moved yet at t
RS = (RS.join(BAL.select("cust_pwr_id", "m_idx", "bal_med12", "bal_now"),
              ["cust_pwr_id", "m_idx"], "left")
        .withColumn("d_atrisk", ((F.coalesce("bal_med12", F.lit(0.0)) <= BAL_FLOOR) |
                                 (F.col("bal_now") >= MOVE_ATRISK_FRAC*F.col("bal_med12")))
                    .cast("int"))
        .drop("bal_med12", "bal_now"))

is_pos = (F.col("event_A").between(F.col("m_idx")+1, F.col("m_idx")+max(HORIZON_GRID)) |
          F.col("event_D").between(F.col("m_idx")+1, F.col("m_idx")+max(HORIZON_GRID)))
TR_RAW = collect_pd(RS.filter(F.col("m_idx") <= max(ORIGINS) - min(HORIZON_GRID))
                    .withColumn("_u", (F.abs(F.hash(F.concat_ws("|", "cust_pwr_id",
                                F.col("m_idx").cast("string"), F.lit(SEED)))) % 100000)/100000.0)
                    .filter(is_pos | (F.col("_u") < NEG_SAMPLE)), "TRAIN")
TE_RAW = {t: collect_pd(RS.filter(F.col("m_idx") == t), f"TEST {t}") for t in ORIGINS}

def prep(d):
    d = d.copy()
    for c in d.columns:
        if c.startswith("ld_"):   d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0)
        elif c.startswith("md_"): d[c] = pd.to_numeric(d[c], errors="coerce").fillna(1.0)
    for c in ["bar_now", "bar_med12", "bar_peak12"]:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0).clip(lower=0)
    d["d_atrisk"] = pd.to_numeric(d.get("d_atrisk", 1), errors="coerce").fillna(1).astype(int)
    return d
TRp = prep(TR_RAW); TEp = {k: prep(v) for k, v in TE_RAW.items()}

def label(d, defn, H):
    """Forward window only. A row whose panel ends before t+H with no event is
    UNOBSERVABLE and is dropped — zeroing it relabels every censored client a
    stayer."""
    ev = pd.to_numeric(d["event_A" if defn == "A_full_exit" else "event_D"], errors="coerce")
    t = pd.to_numeric(d["m_idx"], errors="coerce")
    y = ((ev > t) & (ev <= t + H)).astype(float)
    obs = (t + H <= M_MAX) | (y == 1)
    keep = obs & ((ev.isna()) | (ev > t))
    if defn == "D_money_move": keep = keep & (d["d_atrisk"] == 1)
    o = d.loc[keep].copy(); o["y"] = y.loc[keep].values; o["event_m"] = ev.loc[keep].values
    return o
print(f"  train {len(TRp):,} rows · test {sum(len(v) for v in TEp.values()):,} rows")


In [ ]:
# =====================================================================
# 5 · FIT EVERY SET AT EVERY HORIZON + PERMUTATION CONTROL [OUTPUT BLOCK 4]
# =====================================================================
# Calibration slice: the last CAL_MONTHS of each fold's TRAINING window are
# held out of the model fit and used only to fit the isotonic map. They are
# still <= T-H, so nothing leaks.
CAL_MONTHS = 2
def run_all(defn):
    folds, scored = [], []
    for H in HORIZON_GRID:
        for T in ORIGINS:
            if T + H > M_MAX: continue
            trf = label(TRp[TRp.m_idx <= T - H], defn, H)
            if len(trf) == 0 or trf.y.sum() < MIN_TRAIN_POS: continue
            cut = trf.m_idx.max() - CAL_MONTHS
            fit_df, cal_df = trf[trf.m_idx <= cut], trf[trf.m_idx > cut]
            if len(cal_df) < 5000 or cal_df.y.sum() < 20:
                fit_df, cal_df = trf, trf          # degrade rather than fail
            te = label(TEp[T], defn, H)
            if len(te) == 0 or te.y.sum() < 1: continue
            out = te[["cust_pwr_id", "m_idx", "y", "event_m",
                      "bar_now", "bar_med12"]].copy()
            out["H"], out["defn"] = H, defn
            for fs, cols in FSETS.items():
                cols = [c for c in cols if c in fit_df.columns]
                sp = fit_spec(fit_df, cols)
                p_raw = predict_p(sp, te)
                knots = pav(predict_p(sp, cal_df), cal_df.y.values)
                p_cal = apply_iso(knots, p_raw)
                out[f"p_{fs}"] = p_cal
                out[f"praw_{fs}"] = p_raw
                folds.append(dict(defn=defn, H=H, origin=T, feature_set=fs,
                                  n_feat=len(sp["cols"]) if sp else 0,
                                  n_train=len(fit_df), n_pos=int(trf.y.sum()),
                                  base=float(te.y.mean()),
                                  auc=auc(te.y.values, p_raw),
                                  auc_cal=auc(te.y.values, p_cal)))
            scored.append(out)
    return pd.DataFrame(folds), (pd.concat(scored, ignore_index=True) if scored
                                 else pd.DataFrame())

t0 = time.time()
FOLD, SCORED = {}, {}
for defn in LABELS_RUN:
    f, s = run_all(defn); FOLD[defn], SCORED[defn] = f, s
    print(f"  {defn}: {len(f):,} fold-rows, {len(s):,} scored rows "
          f"({time.time()-t0:,.0f}s)")
F_ALL = pd.concat(FOLD.values(), ignore_index=True)
F_ALL.to_csv(OUT_DIR / "v10_folds.csv", index=False)

A = (F_ALL[F_ALL.defn == PRIMARY_DEF].groupby(["feature_set", "H"], as_index=False)
     .agg(auc=("auc", "mean"), sd=("auc", "std"), base=("base", "mean"),
          n_feat=("n_feat", "max")))
piv = A.pivot_table(index="feature_set", columns="H", values="auc").reindex(FS_ORDER)
disp(piv.round(4).reset_index(),
     title=f"5a &middot; <b>AUC by feature set and horizon</b>, {PRIMARY_DEF}, "
           f"{len(ORIGINS)} rolling origins", save="v10_auc_grid")
lines(piv.T, "5a · Discrimination by horizon and feature set",
      f"{PRIMARY_DEF} · rolling origin · AUC on the full test book",
      ylab="AUC", xlab="prediction horizon (months)", colors=FSCOL,
      save="v10_auc_by_horizon")

if "D_money_move" in FOLD and len(FOLD["D_money_move"]):
    D = (FOLD["D_money_move"].groupby(["feature_set", "H"], as_index=False)
         .agg(auc=("auc", "mean")))
    pd_ = D.pivot_table(index="feature_set", columns="H", values="auc").reindex(FS_ORDER)
    disp(pd_.round(4).reset_index(),
         title="5b &middot; The same for <code>D_money_move</code>. <b>Read the deposit-only row "
               "with the circularity caveat from §2</b> — the label is defined on the balance",
         save="v10_auc_grid_D")

# ── permutation control: the fit must not survive shuffled labels ─────
if RUN_AUDIT:
    rng = np.random.default_rng(SEED)
    T0, H0 = ORIGINS[len(ORIGINS)//2], PRIMARY_H
    trf = label(TRp[TRp.m_idx <= T0-H0], PRIMARY_DEF, H0)
    te  = label(TEp[T0], PRIMARY_DEF, H0)
    rows = []
    for fs, cols in FSETS.items():
        cols = [c for c in cols if c in trf.columns]
        real = fit_spec(trf, cols)
        sh = trf.copy(); sh["y"] = rng.permutation(sh.y.values)
        perm = fit_spec(sh, cols)
        rows.append(dict(feature_set=fs,
                         auc_real=auc(te.y.values, predict_p(real, te)),
                         auc_permuted_labels=auc(te.y.values, predict_p(perm, te))))
    P = pd.DataFrame(rows)
    P["passes"] = np.where(P.auc_permuted_labels.between(0.44, 0.56), "yes", "NO — INVESTIGATE")
    disp(P.round(4), title=f"5c &middot; <b>Label-permutation control</b> at m_idx {T0}, H={H0}. "
         "Training labels are shuffled and the untouched test month is scored. "
         "<b>AUC must collapse to ~0.50</b>", save="v10_permutation")
    _m = float(P.auc_permuted_labels.mean())
    for r in P.itertuples():
        if not 0.40 <= r.auc_permuted_labels <= 0.60:
            print(f"  WARNING {r.feature_set}: permuted AUC {r.auc_permuted_labels:.3f} "
                  f"— single fold, check before relying on it")
    assert 0.45 <= _m <= 0.55, (
        f"permutation control failed (mean {_m:.3f}) — something carries the label "
        f"across the train/test split")
    note("LEAK", "Is the train/test separation defendable?",
         "window assertion passed; permutation AUC " +
         ", ".join(f"{r.feature_set} {r.auc_permuted_labels:.3f}" for r in P.itertuples()),
         "Rolling origin trains only on rows whose labels resolve before the test month; "
         "shuffling those labels destroys all signal, which is what it should do.")


In [ ]:
# =====================================================================
# 6 · CALIBRATION — before and after the isotonic fix     [OUTPUT BLOCK 5]
# =====================================================================
# v9's top predicted decile was 41% UNDER-predicted, and the top decile IS
# the queue. Expected value multiplies probability by dollars, so the scale
# has to be right, not just the order.
S = SCORED[PRIMARY_DEF]
S = S[S.H == PRIMARY_H].copy()
rows = []
for fs in FS_ORDER:
    for tag, col in [("raw", f"praw_{fs}"), ("isotonic", f"p_{fs}")]:
        d = S[["y", col]].dropna()
        d["dec"] = pd.qcut(d[col].rank(method="first"), 10, labels=False) + 1
        g = d.groupby("dec", as_index=False).agg(pred=(col, "mean"), real=("y", "mean"),
                                                 n=("y", "size"))
        err = float(np.abs(g.real-g.pred).sum()/g.real.sum())
        top = g[g.dec == 10].iloc[0]
        rows.append(dict(feature_set=fs, version=tag, calib_error=err,
                         top_decile_pred=float(top.pred), top_decile_real=float(top.real),
                         top_decile_ratio=float(top.real/top.pred)))
CAL = pd.DataFrame(rows)
disp(CAL.round(4), title="6a &middot; <b>Calibration, raw against isotonic.</b> "
     "<code>top_decile_ratio</code> is the one that matters — that decile is the queue",
     n=20, save="v10_calibration")

if HAVE_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 4.0), sharey=True)
    for ax, tag, col_t in zip(axes, ["raw", "isotonic"], ["praw_", "p_"]):
        for fs in FS_ORDER:
            d = S[["y", col_t+fs]].dropna().rename(columns={col_t+fs: "p"})
            d["dec"] = pd.qcut(d.p.rank(method="first"), 10, labels=False)+1
            g = d.groupby("dec").agg(pred=("p", "mean"), real=("y", "mean"))
            ax.plot(g.pred, g.real, marker="o", ms=4, lw=1.6, color=FSCOL[fs], label=fs)
        lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
        ax.plot([0, lim], [0, lim], ls=":", color=INK, lw=1)
        ax.set_title(tag, fontsize=9.5); ax.set_xlabel("predicted")
    axes[0].set_ylabel("realised"); axes[0].legend(frameon=False, fontsize=7.5)
    _fin(fig, "6b · Calibration before and after the isotonic fit",
         "fitted on the last 2 training months of each fold — still inside t ≤ T−H, so no leakage",
         save="v10_calibration_plot")
_worst = CAL[CAL.version == "isotonic"].top_decile_ratio
note("CALIB", "Is the probability usable for expected-value ranking?",
     f"top-decile ratio after isotonic: {_worst.min():.2f}–{_worst.max():.2f}",
     "The isotonic map is fitted on a held-out slice of the TRAINING window, so it does not see "
     "the test month. A ratio near 1.0 means dollars can be multiplied by the probability.")


## 7 · The savings engine

### The rules
1. Score every at-risk client every calendar month. Rank within the month. Work the top **K**.
2. A client re-flagged within three months is the **same conversation**, not a new one.
3. The RM calls. **Success applies only to clients who were genuinely going to leave** — a call on
   a stayer costs time and saves nothing.
4. On success the deposit balance **freezes at that moment**: we retain `bal_now` at the month of
   the first alert. Anything already gone is already gone, so the decay curve is inside the number
   rather than applied as a haircut afterwards.

```
saved = Σ over first-alerted true attriters of  bal_now(c, t_first) × p_save
cost  = distinct conversations × RM cost per call
```

### The grid
Feature set × alert volume × save rate, with save rates down to **1%**. The point of going that
low is that if the case holds at 1% it does not depend on believing anything about how persuasive
a relationship manager is.

In [ ]:
# =====================================================================
# 7 · SAVINGS: feature set x capacity x save rate        [OUTPUT BLOCK 6]
# =====================================================================
def first_alerts(df, score_col, K, alpha=0.0, bar_col="bar_med12"):
    d = df.copy()
    s = pd.to_numeric(d[score_col], errors="coerce")
    if alpha > 0:
        s = s * np.power(np.maximum(pd.to_numeric(d[bar_col], errors="coerce").fillna(0.0),
                                    1.0), alpha)
    d["_s"] = s
    d["_r"] = d.groupby("m_idx")["_s"].rank(ascending=False, method="first", na_option="bottom")
    fl = d[d._r <= K]
    # drop_duplicates keeps the WHOLE first row; groupby.first() skips NaN per
    # column and would mix values from different months
    first = fl.sort_values(["cust_pwr_id", "m_idx"]).drop_duplicates("cust_pwr_id", keep="first")
    return fl, first

if RUN_SAVINGS:
    t0 = time.time()
    SS = SCORED[PRIMARY_DEF]; SS = SS[SS.H == PRIMARY_H].copy()
    # the incumbent, for reference: it flags on its own schedule, not a top-K
    rows = []
    for fs in FS_ORDER:
        for alpha in VALUE_ALPHAS:
            for K in CAPACITY:
                fl, first = first_alerts(SS, f"p_{fs}", K, alpha)
                tp = first[first.y == 1]
                reach = float(tp.bar_now.sum()); conv = len(first)
                for ps in P_SAVE_GRID:
                    saved = reach*ps; cost = conv*RM_COST_PER_CALL
                    rows.append(dict(feature_set=fs, alpha=alpha, k=K, p_save=ps,
                                     conversations=conv, tp_clients=int(len(tp)),
                                     precision=float(len(tp)/max(conv, 1)),
                                     dollars_reached=reach,
                                     dollars_saved=saved,
                                     saved_annualised=saved*ANNUALISE,
                                     rm_cost=cost, net=saved-cost,
                                     net_annualised=(saved-cost)*ANNUALISE,
                                     saved_per_conversation=saved/max(conv, 1),
                                     breakeven_p_save=cost/max(reach, 1),
                                     median_lead=float((tp.event_m-tp.m_idx).median())
                                                 if len(tp) else np.nan))
    SAV = pd.DataFrame(rows)
    SAV.to_csv(OUT_DIR / "v10_savings.csv", index=False)
    print(f"  savings grid: {len(SAV):,} rows in {time.time()-t0:,.0f}s")

    # ── THE HEADLINE: feature set x save rate, at the base capacity ──
    B = SAV[(SAV.k == QUEUE_K) & (SAV.alpha == ALPHA_BASE)]
    H1 = B.pivot_table(index="feature_set", columns="p_save",
                       values="saved_annualised").reindex(FS_ORDER)
    disp(H1.apply(lambda c: c.map(usd)).reset_index(),
         title=f"7a &middot; <b>Annualised dollars retained</b> — feature set &times; RM save "
               f"rate, at {QUEUE_K:,} alerts a month (&alpha;={ALPHA_BASE})",
         save="v10_headline_grid")
    heat(H1, "7a · Annualised dollars retained by feature set and save rate",
         f"{QUEUE_K:,} alerts a month · α={ALPHA_BASE} · {PRIMARY_DEF} · H={PRIMARY_H}",
         xlab="RM save rate", ylab="", save="v10_heat_fs_psave")
    bar_grouped(H1.T, "RM save rate", FS_ORDER,
                "7b · Even at a 1% save rate the feature sets separate clearly",
                f"annualised retained dollars · {QUEUE_K:,} alerts/month · α={ALPHA_BASE}",
                ylab="annualised $ retained", colors=FSCOL, save="v10_bar_fs_psave")

    # ── the uplift: what the payment work is worth ──────────────────
    UP = pd.DataFrame({
        "over deposit_only: payment_only": H1.loc["payment_only"] - H1.loc["deposit_only"],
        "over deposit_only: both":         H1.loc["both"] - H1.loc["deposit_only"],
        "over deposit_only: all_features": H1.loc["all_features"] - H1.loc["deposit_only"],
    })
    disp(UP.apply(lambda c: c.map(usd)).reset_index().rename(columns={"index": "p_save"}),
         title="7c &middot; <b>The uplift over a deposit-only queue</b> — this is what the "
               "payment work is worth, in dollars, at each save rate", save="v10_uplift")
    MULT = pd.DataFrame({fs: H1.loc[fs]/H1.loc["deposit_only"] for fs in FS_ORDER})
    disp(MULT.round(2).reset_index().rename(columns={"index": "p_save"}),
         title="7d &middot; The same as a multiple of the deposit-only queue "
               "(constant across save rates by construction — the ratio is the point)",
         save="v10_multiple")

    # ── capacity: how many calls? ───────────────────────────────────
    C = SAV[(SAV.p_save == P_SAVE_BASE) & (SAV.alpha == ALPHA_BASE)]
    H2 = C.pivot_table(index="k", columns="feature_set",
                       values="saved_annualised")[FS_ORDER]
    lines(H2, "7e · Retained dollars against alert volume",
          f"RM save rate {P_SAVE_BASE:.0%} · α={ALPHA_BASE} · annualised",
          ylab="annualised $ retained", xlab="alerts per month", fmt=usd,
          colors=FSCOL, logx=True, save="v10_savings_by_k")

    mar = H2.copy()
    for fs in FS_ORDER:
        conv = C[C.feature_set == fs].set_index("k").conversations
        mar[fs] = H2[fs].diff()/conv.diff()
    disp(mar.round(0).reset_index(),
         title="7f &middot; <b>Marginal dollars retained per additional conversation.</b> "
               "This is the answer to &ldquo;how many calls should we make&rdquo; — stop where "
               f"it falls below the {usd(RM_COST_PER_CALL)} cost of a call",
         save="v10_marginal")
    lines(mar.dropna(), "7f · Marginal return per extra conversation",
          f"stop where the curve crosses the {usd(RM_COST_PER_CALL)} cost line",
          ylab="$ retained per extra conversation", xlab="alerts per month",
          fmt=usd, colors=FSCOL, logx=True, save="v10_marginal_plot")

    # ── the break-even argument ─────────────────────────────────────
    BE = (SAV[(SAV.alpha == ALPHA_BASE) & (SAV.p_save == P_SAVE_BASE)]
          .pivot_table(index="k", columns="feature_set",
                       values="breakeven_p_save")[FS_ORDER])
    disp((BE*100).round(4).reset_index(),
         title="7g &middot; <b>Break-even save rate (%)</b> — the success rate at which the calls "
               "pay for themselves. <b>Argue about this number, not about the savings figure</b>, "
               "because it is derived from reach and cost rather than assumed",
         save="v10_breakeven")

    _b = SAV[(SAV.feature_set == "all_features") & (SAV.k == QUEUE_K) &
             (SAV.alpha == ALPHA_BASE) & (SAV.p_save == 0.01)].iloc[0]
    _d = SAV[(SAV.feature_set == "deposit_only") & (SAV.k == QUEUE_K) &
             (SAV.alpha == ALPHA_BASE) & (SAV.p_save == 0.01)].iloc[0]
    kv([("alert volume", f"{QUEUE_K:,} a month"),
        ("RM save rate assumed", "1% — the most pessimistic case tested"),
        ("all features · departing clients reached", f"{int(_b.tp_clients):,}"),
        ("all features · defendable dollars in front of an RM", usd(_b.dollars_reached)),
        ("all features · retained, annualised", usd(_b.saved_annualised)),
        ("all features · RM cost", usd(_b.rm_cost)),
        ("all features · net, annualised", usd(_b.net_annualised)),
        ("deposit only · retained, annualised", usd(_d.saved_annualised)),
        ("uplift from the payment work, annualised",
         usd(_b.saved_annualised - _d.saved_annualised)),
        ("break-even save rate for the full set", f"{_b.breakeven_p_save:.3%}")],
       title="7h &middot; <b>The case at its weakest assumption</b>", save="v10_case_1pct")
    note("VALUE", "What is the payment work worth?",
         f"{usd(_b.saved_annualised - _d.saved_annualised)} a year over a deposit-only queue "
         f"at a 1% save rate",
         "Same alert volume, same clients, same folds, same freeze assumption. The only "
         "difference is what the model is allowed to see.")


## 8 · The lead-time ceiling

§7 measures what the queue is worth **as it times today**. The decay curve says timing is the
binding constraint, so this section separates *timing* from *detection*.

Take the clients the queue actually caught and ask a single counterfactual: **if the same clients
had been reached L months before their event, how much would have been on the books?** No
assumption about whether the model could have found them then — only how much money was still
there. That is the ceiling on what better early detection can buy, and the gap between it and §7
is the headroom.

In [ ]:
# =====================================================================
# 8 · WHAT EARLIER DETECTION WOULD BE WORTH              [OUTPUT BLOCK 7]
# =====================================================================
if RUN_LEAD:
    SS = SCORED[PRIMARY_DEF]; SS = SS[SS.H == PRIMARY_H]
    rows = []
    for fs in FS_ORDER:
        fl, first = first_alerts(SS, f"p_{fs}", QUEUE_K, ALPHA_BASE)
        tp = first[first.y == 1][["cust_pwr_id", "m_idx", "event_m", "bar_now"]]
        if not len(tp): continue
        # build from plain python tuples — createDataFrame on a pandas frame
        # with numpy dtypes is fragile with arrow disabled
        caught = spark.createDataFrame(
            [(str(a), int(b)) for a, b in zip(tp.cust_pwr_id, tp.event_m)],
            ["cust_pwr_id", "event_m"])
        BB = (BAL.select("cust_pwr_id", "m_idx", "bal_now")
              .join(F.broadcast(caught), "cust_pwr_id", "inner")
              .withColumn("rel_m", F.col("m_idx")-F.col("event_m")))
        got = collect_pd(BB.filter(F.col("rel_m").isin([-l for l in LEAD_GRID]))
                         .groupBy("rel_m").agg(F.sum("bal_now").alias("dollars"),
                                               F.count("*").alias("n")),
                         f"lead ladder {fs}")
        actual_lead = float((tp.event_m - tp.m_idx).median())
        actual_dollars = float(tp.bar_now.sum())
        for r in got.itertuples():
            rows.append(dict(feature_set=fs, lead_months=int(-r.rel_m),
                             clients=int(r.n), defendable=float(r.dollars),
                             actual_median_lead=actual_lead,
                             actual_defendable=actual_dollars))
    LEAD = pd.DataFrame(rows)
    LEAD.to_csv(OUT_DIR / "v10_lead_ceiling.csv", index=False)
    P8 = LEAD.pivot_table(index="lead_months", columns="feature_set",
                          values="defendable").reindex(sorted(LEAD.lead_months.unique()))
    P8 = P8[[c for c in FS_ORDER if c in P8.columns]]
    disp(P8.apply(lambda c: c.map(usd)).reset_index(),
         title=f"8a &middot; <b>Defendable dollars if the same caught clients had been reached "
               f"L months before the event.</b> Their actual median lead today is "
               f"{LEAD.actual_median_lead.iloc[0]:.0f} month(s)", save="v10_lead_ladder")
    lines(P8, "8a · What earlier detection is worth, holding detection fixed",
          f"same caught clients, {QUEUE_K:,} alerts/month, α={ALPHA_BASE} · balance still on "
          f"book at each lead", ylab="defendable $", xlab="months of lead", fmt=usd,
          colors=FSCOL, save="v10_lead_plot")

    _fs = "all_features"
    base = float(LEAD[(LEAD.feature_set == _fs)].actual_defendable.iloc[0])
    tbl = LEAD[LEAD.feature_set == _fs].set_index("lead_months").defendable
    kv([("actual median lead today", f"{LEAD.actual_median_lead.iloc[0]:.0f} months"),
        ("defendable at today's timing", usd(base)),
        *[(f"if reached at {l} months' lead", usd(tbl.get(l, np.nan))) for l in LEAD_GRID],
        ("headroom from -2 to -6 lead",
         usd(float(tbl.get(6, np.nan)) - float(tbl.get(2, np.nan)))),
        ("that headroom, at a 1% save rate, annualised",
         usd((float(tbl.get(6, np.nan)) - float(tbl.get(2, np.nan)))*0.01*ANNUALISE))],
       title="8b &middot; <b>Timing, not detection, is the binding constraint</b>",
       save="v10_lead_headline")
    note("LEAD", "How much is earlier detection worth?",
         f"{usd(float(tbl.get(6, np.nan)) - float(tbl.get(2, np.nan)))} of extra defendable "
         f"balance moving the alert from 2 to 6 months' lead",
         "Holds detection fixed — the same clients, reached earlier. This is the ceiling on "
         "what a longer-horizon or earlier-firing model can buy.")


In [ ]:
# =====================================================================
# 9 · ROBUSTNESS                                         [OUTPUT BLOCK 8]
# =====================================================================
if RUN_ROBUST:
    SS = SCORED[PRIMARY_DEF]; SS = SS[SS.H == PRIMARY_H]
    # ── (a) is the value result a bet on a handful of whales? ────────
    big = (SS.groupby("cust_pwr_id", as_index=False).bar_med12.max()
           .sort_values("bar_med12", ascending=False))
    rows = []
    for drop in JACKKNIFE_TOP:
        excl = set(big.cust_pwr_id.head(drop))
        sub = SS[~SS.cust_pwr_id.isin(excl)]
        for fs in ["deposit_only", "all_features"]:
            for alpha in VALUE_ALPHAS:
                fl, first = first_alerts(sub, f"p_{fs}", QUEUE_K, alpha)
                tp = first[first.y == 1]
                rows.append(dict(dropped_top=drop, feature_set=fs, alpha=alpha,
                                 tp_clients=int(len(tp)),
                                 dollars_reached=float(tp.bar_now.sum())))
    JK = pd.DataFrame(rows)
    piv = JK[JK.feature_set == "all_features"].pivot_table(
        index="dropped_top", columns="alpha", values="dollars_reached")
    disp(piv.apply(lambda c: c.map(usd)).reset_index(),
         title="9a &middot; <b>Jackknife.</b> Drop the N largest at-risk clients and re-run. If "
               "the &alpha; advantage collapses, the value queue is a bet on catching individual "
               "whales rather than a property of the ranking", save="v10_jackknife")
    heat(piv, "9a · Dollars reached after dropping the N largest clients",
         f"all_features · {QUEUE_K:,} alerts/month", xlab="α (value weight)",
         ylab="top clients dropped", save="v10_jackknife_heat")
    _r0 = piv.loc[0]; _r = piv.iloc[-1]
    kv([("best alpha with everyone in", float(_r0.idxmax())),
        ("best alpha after dropping the top %d" % JACKKNIFE_TOP[-1], float(_r.idxmax())),
        ("dollars reached, alpha=0, everyone in", usd(_r0.get(0.0))),
        ("dollars reached, best alpha, everyone in", usd(_r0.max())),
        ("...after the drop", usd(_r.max())),
        ("advantage survives?", "yes" if _r.max() > _r.get(0.0, 0)*1.5 else "WEAKENS — investigate")],
       title="9b &middot; Does value ranking survive removing the whales?", save="v10_jk_verdict")

    # ── (b) the B_bal_exit pool the business case currently ignores ──
    att_A = lab.filter(F.col("q_A_full_exit").isNotNull()).select(
        "cust_pwr_id", F.col("q_A_full_exit").alias("event_m"))
    att_B = (lab.filter(F.col("q_B_bal_exit").isNotNull() & F.col("q_A_full_exit").isNull())
             .select("cust_pwr_id", F.col("q_B_bal_exit").alias("event_m")))
    def pool(df, nm):
        r = (BAL.join(df, "cust_pwr_id", "inner")
             .withColumn("rel_m", F.col("m_idx")-F.col("event_m"))
             .filter(F.col("rel_m") == -12)
             .agg(F.count(F.lit(1)).alias("n"), F.sum("bal_med12").alias("pool")).collect()[0])
        return dict(cohort=nm, clients=int(r["n"]), pool=float(r["pool"] or 0))
    PB = pd.DataFrame([pool(att_A, "A_full_exit (counted today)"),
                       pool(att_B, "B_bal_exit only (NOT counted today)")])
    PB.loc[len(PB)] = dict(cohort="combined", clients=PB.clients.sum(), pool=PB.pool.sum())
    disp(usd_col(PB, "pool"),
         title="9c &middot; <b>The pool the business case leaves out.</b> B clients drain without "
               "closing, they are invisible to balance monitoring, and their money leaves too",
         save="v10_pool_B")
    note("POOLB", "How much does ignoring B_bal_exit understate the pool?",
         usd(float(PB[PB.cohort.str.startswith("B_")].pool.iloc[0])),
         "Measured the same way as the A pool — sum of bal_med12 at rel_m -12. Clients with a B "
         "event and no A event only.")

    # ── (c) precision / dollars frontier, all sets and capacities ────
    pts = []
    for fs in FS_ORDER:
        for K in CAPACITY:
            r = SAV[(SAV.feature_set == fs) & (SAV.k == K) & (SAV.alpha == ALPHA_BASE) &
                    (SAV.p_save == P_SAVE_BASE)]
            if not len(r): continue
            r = r.iloc[0]
            pts.append(dict(x=r.precision, y=r.dollars_reached, s=22+K/60,
                            color=FSCOL[fs], label=fs, tag=(f"K={K:,}" if fs == "all_features"
                                                            else None)))
    scatter(pts, "9d · The frontier — precision against dollars reached",
            f"marker size is alert volume · α={ALPHA_BASE} · each point is one feature set "
            f"at one capacity", xlab="precision (share of conversations that were leaving)",
            ylab="defendable dollars reached", fmty=usd, save="v10_frontier")


---

## After this run

1. **Lead the conversation with the break-even, not the savings.** `breakeven_p_save` in 7g is
   derived from reach and cost; the savings grid is derived from an assumption. If break-even is
   around 0.1%, the programme is defensible under any belief about RM effectiveness and the
   argument is finished. The savings grid is then illustration, not evidence.

2. **§7c is the number that answers "was this work worth doing".** Same alert volume, same
   clients, same folds, same freeze assumption — the only difference is what the model is allowed
   to see. Quote it at the 1% save rate, because a number that survives the worst assumption
   cannot be argued down.

3. **§8 reframes the roadmap.** If moving the alert from 2 to 6 months of lead is worth more than
   every feature improvement in v8 and v9 combined, then the next release is about **horizon and
   the `D_money_move` target**, not about more features. Check 5a: if AUC at H=9 and H=12 holds up
   for `all_features`, a longer-horizon queue is available immediately.

4. **Watch the deposit-only row on `D_money_move` (5b).** The label is defined on the balance and
   the at-risk guard only partly removes the overlap. If `deposit_only` beats `payment_only` there
   by much more than it does on `A_full_exit`, treat the gap as construction rather than
   discovery.

5. **If §9a shows the α advantage collapsing** once the top 50–100 clients are dropped, the value
   queue should be reframed: not "rank everyone by expected dollars" but "**a named watch-list of
   the largest relationships, reviewed monthly, plus a probability queue for everyone else**".
   That is a different and arguably better operating model, and the jackknife is what decides it.

6. **Still outstanding.** No live evidence that an alert changes an outcome; the 20% of the deposit
   book invisible to payments has never been profiled; and `p_save` needs a pilot with a holdout
   arm, because a save and a client who was never going to leave look identical in this data.
